In [ ]:
# Punto 7: Análisis con PySpark (API DataFrames)
# Cargar las tablas desde el catálogo Hive (ya están en isabela_db)
ratings = spark.table("isabela_db.ratings")
tmdb = spark.table("isabela_db.tmdb")

# Mostrar esquemas para verificar
print("Esquema de ratings:")
ratings.printSchema()
print("Esquema de tmdb:")
tmdb.printSchema()

# 1. Top 10 películas mejor calificadas (rating promedio)
print("\n1. Top 10 mejor calificadas:")
top_rated = ratings.groupBy("movieId").avg("rating").withColumnRenamed("avg(rating)", "avg_rating") \
    .join(tmdb, ratings["movieId"] == tmdb["tmdbId"], "inner") \
    .select("movieId", "title", "avg_rating") \
    .orderBy("avg_rating", ascending=False).limit(10)
top_rated.show(truncate=False)

# 2. Distribución de ratings (frecuencia por estrella entera)
from pyspark.sql.functions import round as spark_round, count
print("\n2. Distribución de ratings:")
distribucion = ratings.withColumn("star_rating", spark_round("rating")) \
    .groupBy("star_rating").agg(count("*").alias("frequency")) \
    .orderBy("star_rating")
distribucion.show()

# 3. Películas más populares (mayor número de ratings)
print("\n3. Top 10 más populares (por número de ratings):")
populares = ratings.groupBy("movieId").count().withColumnRenamed("count", "num_ratings") \
    .orderBy("num_ratings", ascending=False).limit(10)
populares.show()

# 4. Evolución temporal por mes (usando timestamp)
from pyspark.sql.functions import from_unixtime, year, month, avg, count as cnt
print("\n4. Evolución temporal (promedio mensual de ratings):")
ratings_with_date = ratings.withColumn("date", from_unixtime(ratings["timestamp"].cast("long"))) \
    .withColumn("year", year("date")) \
    .withColumn("month", month("date"))
temporal = ratings_with_date.groupBy("year", "month").agg(
    avg("rating").alias("avg_rating"),
    cnt("*").alias("total_ratings")
).orderBy("year", "month")
temporal.show(20)

# 5. Relación presupuesto vs rating promedio (top 20 presupuesto)
print("\n5. Top 20 películas con mayor presupuesto y su rating promedio:")
# Primero unimos tmdb y ratings, luego agrupamos por película, calculamos avg rating y número de ratings
joined = tmdb.join(ratings, tmdb["tmdbId"] == ratings["movieId"], "inner") \
    .filter(tmdb["budget"] > 0) \
    .groupBy("tmdbId", "title", "budget") \
    .agg(avg("rating").alias("avg_rating"), cnt("*").alias("num_ratings"))
top_budget = joined.orderBy("budget", ascending=False).limit(20)
top_budget.show(truncate=False)

# Coeficiente de correlación entre budget y avg_rating (solo las películas con ratings)
from pyspark.sql.functions import corr
corr_value = joined.select(corr("budget", "avg_rating")).collect()[0][0]
print(f"\nCoeficiente de correlación (Pearson) entre presupuesto y rating promedio: {corr_value:.4f}")

In [ ]:
# =============================================
# PUNTO 7: ANÁLISIS ESTADÍSTICO CON PYSPARK
# =============================================

from pyspark.sql.functions import col, avg, count, round as spark_round, year, month, from_unixtime, corr

# Cargar DataFrames desde las tablas Hive
ratings = spark.table("isabela_db.ratings")
tmdb = spark.table("isabela_db.tmdb")

print("=== 1. VERIFICACIÓN DE ESQUEMAS ===")
print("Columnas de ratings:", ratings.columns)
ratings.printSchema()
print("Columnas de tmdb:", tmdb.columns)
tmdb.printSchema()



# -------------------------------------------------------------------
# 3. ANÁLISIS COMPLEMENTARIO (sin gráficos)
# -------------------------------------------------------------------
print("\n=== 3. ANÁLISIS COMPLEMENTARIO ===")

# 3.1 Correlación entre vote_average (TMDB) y rating promedio de usuarios
comparison = tmdb.join(ratings.groupBy("movieid").agg(avg("rating").alias("avg_rating")),
                       tmdb["tmdbid"] == ratings["movieid"], "inner") \
    .select("title", "vote_average", "avg_rating")
corr_vote = comparison.select(corr("vote_average", "avg_rating")).collect()[0][0]
print(f"Correlación entre vote_average (TMDB) y rating de usuarios: {corr_vote:.4f}")

# 3.2 Estadísticas descriptivas de ratings (describe)
print("\nEstadísticas descriptivas de ratings:")
ratings.describe().show()

# 3.3 Top 10 películas según vote_average (TMDB)
print("\nTop 10 películas mejor valoradas por TMDB (vote_average):")
tmdb.orderBy(col("vote_average").desc()).select("title", "vote_average", "budget", "revenue").show(10)

# 3.4 Películas con mayor recaudación (revenue)
print("\nTop 10 películas con mayor recaudación:")
tmdb.orderBy(col("revenue").desc()).select("title", "revenue", "budget").show(10)

# 3.5 Películas con mejor ROI (revenue/budget) - evitando división por cero
from pyspark.sql.functions import when
roi_df = tmdb.withColumn("roi", col("revenue") / when(col("budget") > 0, col("budget")).otherwise(1))
print("\nTop 10 películas con mejor ROI (retorno sobre inversión):")
roi_df.orderBy(col("roi").desc()).select("title", "budget", "revenue", "roi").show(10)

print("\n✅ Análisis completado. Los resultados numéricos y tablas anteriores responden todas las preguntas de negocio.")